## First test the SQL runner independently

In [2]:
import sys
from pathlib import Path


project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


from src.sql_runner import (
    read_sql_file,
    split_sql_statements,
)


bridge_sql_path = (
    project_root
    / "sql"
    / "03_olap"
    / "05_sync_bridge_title_genre.sql"
)


bridge_sql = read_sql_file(
    bridge_sql_path
)

statements = split_sql_statements(
    bridge_sql
)


print(
    "Statements detected:",
    len(statements),
)

for index, statement in enumerate(
    statements,
    start=1,
):
    print(
        f"Statement {index}:",
        statement[:80],
        "...",
    )

Statements detected: 2
Statement 1: -- ============================================================================
 ...
Statement 2: -- Rebuild relationships from the current OLTP state.
INSERT INTO analytics.brid ...


### Inspect ETL Tracking Tables

Before executing the production pipeline runners, inspect the ETL schema to identify the table responsible for tracking ingestion batches.

The batch metadata will allow the pipeline to use the correct `batch_id` instead of relying on hard-coded values.

In [3]:
from src.database import get_etl_connection


# Inspect the ETL schema for available tracking tables.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT
                table_schema,
                table_name
            FROM information_schema.tables
            WHERE table_schema = 'etl'
            ORDER BY table_name;
            """
        )

        etl_tables = cursor.fetchall()


# Display the available ETL tables.
for table in etl_tables:
    print(table)

('etl', 'batch_history')
('etl', 'pipeline_control')
('etl', 'quality_results')


### Inspect Batch Tracking Metadata

The `etl.batch_history` table stores metadata about ingestion batches processed by the pipeline.

Before running the OLAP transformation runner, inspect its structure and recent records so that the transformation can use the correct batch identifier from the ETL control layer rather than relying on a hard-coded value.

In [4]:
# Inspect the structure of the ETL batch history table.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT
                column_name,
                data_type
            FROM information_schema.columns
            WHERE table_schema = 'etl'
              AND table_name = 'batch_history'
            ORDER BY ordinal_position;
            """
        )

        batch_history_columns = cursor.fetchall()


# Display the batch tracking fields.
print("etl.batch_history columns:")

for column in batch_history_columns:
    print(column)

etl.batch_history columns:
('batch_id', 'bigint')
('pipeline_name', 'character varying')
('file_name', 'text')
('file_checksum', 'text')
('received_at', 'timestamp with time zone')
('processing_started_at', 'timestamp with time zone')
('processing_completed_at', 'timestamp with time zone')
('rows_received', 'bigint')
('rows_processed', 'bigint')
('status', 'character varying')
('error_message', 'text')


### Inspect Recent ETL Batches

Review the most recent batch records to determine which completed ingestion batch should be supplied to the OLAP transformation pipeline.

This links the transformation layer to the ingestion metadata and helps preserve traceability between pipeline stages.

In [5]:
# Retrieve the most recent ingestion batches.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT *
            FROM etl.batch_history
            ORDER BY 1 DESC
            LIMIT 10;
            """
        )

        recent_batches = cursor.fetchall()


# Display recent batch metadata.
print("Recent ETL batches:")

for batch in recent_batches:
    print(batch)

Recent ETL batches:
(13, 'netflix_incremental_pipeline', 'credits.csv', '7122fdc347134241a3192436ae1823ed3e2a50e57985e10ee337cd86b260d831', datetime.datetime(2026, 8, 19, 9, 36, 35, 48395, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), datetime.datetime(2026, 8, 19, 9, 37, 25, 321016, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), datetime.datetime(2026, 8, 21, 22, 5, 3, 413501, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), 77801, None, 'SUCCESS', None)
(12, 'netflix_incremental_pipeline', 'titles.csv', '639cba13a200033e1ebb8e71243eef2d6d4453706ccff67e101f4935dbaad012', datetime.datetime(2026, 8, 19, 9, 32, 53, 604618, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), datetime.datetime(2026, 8, 19, 9, 32, 53, 691171, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), datetime.datetime(2026, 8, 21, 22, 5, 3, 390974, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), 5850, None, 'SUCCESS', None)
(11, 'netflix_incremental_pipeline', 'titles.csv', '639cba13a200033e1ebb8e71243eef2d6d4453706ccff67e101

### Inspect Batch Processing Status

Check the latest `titles.csv` and `credits.csv` batch records and their processing states.

This confirms whether the batches have already been marked as `SUCCESS`, or whether the current test data is still registered under another pipeline state such as `RECEIVED`, `RUNNING`, or `FAILED`.

In [6]:
# Inspect the latest titles and credits batch records.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT
                batch_id,
                file_name,
                rows_received,
                rows_processed,
                status,
                processing_started_at,
                processing_completed_at,
                error_message
            FROM etl.batch_history
            WHERE pipeline_name = 'netflix_incremental_pipeline'
              AND file_name IN ('titles.csv', 'credits.csv')
            ORDER BY batch_id DESC;
            """
        )

        batch_status_records = cursor.fetchall()


# Display the current processing state of each source batch.
print("Current source batch states:")

for record in batch_status_records:
    print(record)

Current source batch states:
(13, 'credits.csv', 77801, None, 'SUCCESS', datetime.datetime(2026, 8, 19, 9, 37, 25, 321016, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), datetime.datetime(2026, 8, 21, 22, 5, 3, 413501, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), None)
(12, 'titles.csv', 5850, None, 'SUCCESS', datetime.datetime(2026, 8, 19, 9, 32, 53, 691171, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), datetime.datetime(2026, 8, 21, 22, 5, 3, 390974, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), None)
(11, 'titles.csv', 5850, None, 'RUNNING', datetime.datetime(2026, 8, 19, 2, 24, 54, 850013, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), None, None)
(10, 'titles.csv', 5850, None, 'RUNNING', datetime.datetime(2026, 8, 19, 2, 24, 46, 3981, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), None, None)
(9, 'titles.csv', 5850, None, 'RUNNING', datetime.datetime(2026, 8, 19, 2, 16, 8, 315102, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), None, None)
(8, 'titles.csv', 5850, None, 'RECEIVE

### Resolve Current Test Batch IDs

Retrieve the most recent real `titles.csv` and `credits.csv` batch identifiers for pipeline-runner testing.

At this stage, the batch lifecycle has not yet been fully automated, so the test resolves the latest registered source batches regardless of final status. The completed Airflow pipeline will later pass the active batch IDs directly between tasks.

In [7]:
# Retrieve the latest registered titles and credits batches.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:

        # Resolve the newest titles source batch.
        cursor.execute(
            """
            SELECT batch_id
            FROM etl.batch_history
            WHERE pipeline_name = 'netflix_incremental_pipeline'
              AND file_name = 'titles.csv'
            ORDER BY batch_id DESC
            LIMIT 1;
            """
        )

        latest_titles_batch = cursor.fetchone()

        # Resolve the newest credits source batch.
        cursor.execute(
            """
            SELECT batch_id
            FROM etl.batch_history
            WHERE pipeline_name = 'netflix_incremental_pipeline'
              AND file_name = 'credits.csv'
            ORDER BY batch_id DESC
            LIMIT 1;
            """
        )

        latest_credits_batch = cursor.fetchone()


# Confirm both required source batches exist.
assert latest_titles_batch is not None, (
    "No titles.csv batch was found."
)

assert latest_credits_batch is not None, (
    "No credits.csv batch was found."
)


# Extract the batch IDs for downstream transformation tests.
titles_batch_id = latest_titles_batch[0]
credits_batch_id = latest_credits_batch[0]


print("Titles batch ID:", titles_batch_id)
print("Credits batch ID:", credits_batch_id)

Titles batch ID: 12
Credits batch ID: 13


### Verify OLTP Title-Genre Synchronisation

Confirm that the production title-to-genre transformation is recognised as a two-statement transaction.

The first statement removes previous relationships for titles in the active batch, while the second rebuilds the relationships from the latest staging data.

In [8]:
# Load the updated OLTP title-genre synchronisation script.
title_genre_sql_path = (
    project_root
    / "sql"
    / "02_oltp"
    / "05_sync_title_genres.sql"
)

title_genre_sql = read_sql_file(
    title_genre_sql_path
)

# Confirm that DELETE and INSERT are detected separately.
title_genre_statements = split_sql_statements(
    title_genre_sql
)

print(
    "Statements detected:",
    len(title_genre_statements),
)

assert len(title_genre_statements) == 2

print("PASS: title-genre synchronisation uses two statements.")

Statements detected: 2
PASS: title-genre synchronisation uses two statements.


### Verify OLTP Title-Country Synchronisation

Confirm that the production title-to-country transformation is recognised as a two-statement transaction.

The first statement removes previous country relationships for titles in the active batch, while the second rebuilds the relationships from the latest staging data.

In [9]:
# Load the updated OLTP title-country synchronisation script.
title_country_sql_path = (
    project_root
    / "sql"
    / "02_oltp"
    / "06_sync_title_countries.sql"
)

title_country_sql = read_sql_file(
    title_country_sql_path
)

# Confirm that DELETE and INSERT are detected separately.
title_country_statements = split_sql_statements(
    title_country_sql
)

print(
    "Statements detected:",
    len(title_country_statements),
)

assert len(title_country_statements) == 2

print("PASS: title-country synchronisation uses two statements.")

Statements detected: 2
PASS: title-country synchronisation uses two statements.


### Run the Production OLTP Transformation

Execute the complete OLTP transformation sequence using the resolved titles and credits batches.

This validates that the production runner can apply all incremental entity, relationship, and credit transformations in the correct order without relying on manual notebook execution.

In [10]:
# Import the production OLTP transformation runner.
from src.oltp_runner import run_oltp_transformations


# Execute the complete OLTP transformation sequence.
oltp_results = run_oltp_transformations(
    titles_batch_id=titles_batch_id,
    credits_batch_id=credits_batch_id,
)


# Display each SQL script executed by the production runner.
for result in oltp_results:
    print(
        result["sql_file"],
        "→ statements:",
        result["statements_executed"],
    )


print("PASS: production OLTP runner completed.")

01_upsert_titles.sql → statements: 1
02_upsert_people.sql → statements: 1
03_upsert_genres.sql → statements: 1
04_upsert_countries.sql → statements: 1
05_sync_title_genres.sql → statements: 2
06_sync_title_countries.sql → statements: 2
07_upsert_credits.sql → statements: 1
PASS: production OLTP runner completed.


### Run the Production OLAP Transformation

Execute the complete OLAP transformation sequence using the resolved titles batch.

This validates that all dimension, bridge, and fact transformations can run through the production OLAP runner instead of being executed manually in notebook cells.

In [13]:

# Run the complete production OLAP transformation pipeline.
from src.olap_runner import run_olap_transformations
olap_results = run_olap_transformations(
    titles_batch_id=titles_batch_id,
)

for result in olap_results:
    print(
        result["sql_file"],
        "→ statements:",
        result["statements_executed"],
    )

print("PASS: production OLAP runner completed.")

01_upsert_dim_title.sql → statements: 1
02_upsert_dim_person.sql → statements: 1
03_upsert_dim_genre.sql → statements: 1
04_upsert_dim_country.sql → statements: 1
05_sync_bridge_title_genre.sql → statements: 2
06_sync_bridge_title_country.sql → statements: 2
07_upsert_fact_credits.sql → statements: 1
08_upsert_fact_title_metrics.sql → statements: 1
PASS: production OLAP runner completed.


### Validate Full OLAP Runner Idempotency

Capture the current warehouse row counts, execute the complete OLAP runner again using the same batch, and confirm that no analytical table gains duplicate records.

In [14]:
# Return row counts for every core OLAP table.
def get_olap_counts():

    olap_tables = [
        "dim_title",
        "dim_person",
        "dim_genre",
        "dim_country",
        "bridge_title_genre",
        "bridge_title_country",
        "fact_credits",
        "fact_title_metrics",
    ]

    counts = {}

    # Read the current warehouse state.
    with get_etl_connection() as connection:
        with connection.cursor() as cursor:

            for table_name in olap_tables:
                cursor.execute(
                    f"""
                    SELECT COUNT(*)
                    FROM analytics.{table_name};
                    """
                )

                counts[table_name] = cursor.fetchone()[0]

    return counts


# Capture the state before the repeated pipeline run.
olap_counts_before_rerun = get_olap_counts()


# Run the exact same production OLAP pipeline again.
run_olap_transformations(
    titles_batch_id=titles_batch_id,
)


# Capture the state after the repeated run.
olap_counts_after_rerun = get_olap_counts()


# Confirm every OLAP table retained the same row count.
for table_name in olap_counts_before_rerun:

    before = olap_counts_before_rerun[table_name]
    after = olap_counts_after_rerun[table_name]

    print(
        table_name,
        "| before:",
        before,
        "| after:",
        after,
    )

    assert before == after, (
        f"{table_name} changed after rerunning the same batch."
    )


print("PASS: complete production OLAP runner is idempotent.")

dim_title | before: 5849 | after: 5849
dim_person | before: 54589 | after: 54589
dim_genre | before: 19 | after: 19
dim_country | before: 109 | after: 109
bridge_title_genre | before: 15088 | after: 15088
bridge_title_country | before: 6528 | after: 6528
fact_credits | before: 77800 | after: 77800
fact_title_metrics | before: 5849 | after: 5849
PASS: complete production OLAP runner is idempotent.


## Build the End-to-End Production Pipeline Coordinator

Create one production entry point that executes the OLTP layer, OLAP layer, and data-quality gate in sequence.

This coordinator represents the business flow that Airflow will later orchestrate as separate tasks while preserving clear failure boundaries and batch lineage.

In [15]:
# Import the end-to-end production coordinator.
from src.pipeline_runner import run_transformation_pipeline


# Execute the full transformation and validation flow.
pipeline_results = run_transformation_pipeline(
    titles_batch_id=titles_batch_id,
    credits_batch_id=credits_batch_id,
)


print("Titles batch:", pipeline_results["titles_batch_id"])
print("Credits batch:", pipeline_results["credits_batch_id"])

print(
    "OLTP transformations:",
    len(pipeline_results["oltp"]),
)

print(
    "OLAP transformations:",
    len(pipeline_results["olap"]),
)

print(
    "Quality checks:",
    len(pipeline_results["quality"]),
)


print("PASS: end-to-end production transformation pipeline completed.")

Titles batch: 12
Credits batch: 13
OLTP transformations: 7
OLAP transformations: 8
Quality checks: 6
PASS: end-to-end production transformation pipeline completed.


### Verify Batch-Level Quality Audit

Confirm that the quality checks executed by the production coordinator were recorded against the active titles batch.

This preserves lineage between the source batch and the validation results produced after transformation.

In [16]:
# Inspect quality results written by the pipeline coordinator.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:

        cursor.execute(
            """
            SELECT
                batch_id,
                check_name,
                status,
                observed_value,
                checked_at
            FROM etl.quality_results
            WHERE pipeline_name = 'netflix_incremental_pipeline'
              AND batch_id = %(batch_id)s
            ORDER BY checked_at DESC;
            """,
            {
                "batch_id": titles_batch_id,
            },
        )

        batch_quality_results = cursor.fetchall()


# Display the quality audit linked to the current batch.
for row in batch_quality_results:
    print(row)

(12, 'missing_title_metrics', 'PASS', 0, datetime.datetime(2026, 8, 22, 0, 5, 39, 154792, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
(12, 'orphan_credit_facts', 'PASS', 0, datetime.datetime(2026, 8, 22, 0, 5, 39, 110067, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
(12, 'orphan_country_bridges', 'PASS', 0, datetime.datetime(2026, 8, 22, 0, 5, 38, 988227, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
(12, 'orphan_genre_bridges', 'PASS', 0, datetime.datetime(2026, 8, 22, 0, 5, 38, 920244, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
(12, 'duplicate_people', 'PASS', 0, datetime.datetime(2026, 8, 22, 0, 5, 38, 866683, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
(12, 'duplicate_titles', 'PASS', 0, datetime.datetime(2026, 8, 22, 0, 5, 38, 681463, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
(12, 'missing_title_metrics', 'PASS', 0, datetime.datetime(2026, 8, 21, 22, 5, 3, 329258, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
(12, 'orphan_credit_facts', 'PASS', 0, datetime.datet

### Validate Successful Batch Lifecycle

Run the complete production transformation pipeline and verify that both source batches are marked `SUCCESS` only after OLTP, OLAP, and all critical quality checks complete successfully.

This confirms that batch metadata now reflects the actual result of the production pipeline.

In [17]:
# Execute the complete production transformation pipeline.
pipeline_results = run_transformation_pipeline(
    titles_batch_id=titles_batch_id,
    credits_batch_id=credits_batch_id,
)


print(
    "Pipeline status:",
    pipeline_results["status"],
)

assert pipeline_results["status"] == "SUCCESS"

print(
    "PASS: production pipeline completed successfully."
)

Pipeline status: SUCCESS
PASS: production pipeline completed successfully.


In [18]:
# Inspect the actual object returned by the production pipeline.
print("Return type:", type(pipeline_results))
print("Pipeline results:")
print(pipeline_results)

# If a dictionary was returned, display the available keys.
if isinstance(pipeline_results, dict):
    print("\nAvailable keys:")
    print(pipeline_results.keys())

Return type: <class 'dict'>
Pipeline results:
{'titles_batch_id': 12, 'credits_batch_id': 13, 'oltp': [{'sql_file': '01_upsert_titles.sql', 'statements_executed': 1}, {'sql_file': '02_upsert_people.sql', 'statements_executed': 1}, {'sql_file': '03_upsert_genres.sql', 'statements_executed': 1}, {'sql_file': '04_upsert_countries.sql', 'statements_executed': 1}, {'sql_file': '05_sync_title_genres.sql', 'statements_executed': 2}, {'sql_file': '06_sync_title_countries.sql', 'statements_executed': 2}, {'sql_file': '07_upsert_credits.sql', 'statements_executed': 1}], 'olap': [{'sql_file': '01_upsert_dim_title.sql', 'statements_executed': 1}, {'sql_file': '02_upsert_dim_person.sql', 'statements_executed': 1}, {'sql_file': '03_upsert_dim_genre.sql', 'statements_executed': 1}, {'sql_file': '04_upsert_dim_country.sql', 'statements_executed': 1}, {'sql_file': '05_sync_bridge_title_genre.sql', 'statements_executed': 2}, {'sql_file': '06_sync_bridge_title_country.sql', 'statements_executed': 2}, {'s

### Reload the Production Pipeline Coordinator

Reload the production coordinator after updating its source file.

Jupyter caches imported Python modules, so reloading ensures the notebook uses the latest implementation without rerunning all previous setup cells.

In [19]:
# Reload the production pipeline module so Jupyter uses the latest code.
import importlib
import src.pipeline_runner as pipeline_runner

importlib.reload(
    pipeline_runner
)

# Rebind the updated production function in the notebook.
run_transformation_pipeline = (
    pipeline_runner.run_transformation_pipeline
)

print("PASS: pipeline coordinator reloaded.")

PASS: pipeline coordinator reloaded.


### Verify the Updated Coordinator Contract

Inspect the current production function to confirm that the successful execution path now adds a final `SUCCESS` status to the returned pipeline metadata.

In [20]:
# Inspect the currently loaded implementation.
import inspect

pipeline_source = inspect.getsource(
    run_transformation_pipeline
)

print(pipeline_source)

# Confirm that the final status is part of the production return contract.
assert 'results["status"] = "SUCCESS"' in pipeline_source

print("PASS: pipeline status return is present.")

def run_transformation_pipeline(
    titles_batch_id,
    credits_batch_id,
):
    """
    Run the complete production transformation pipeline.

    Pipeline control is advanced only after OLTP, OLAP, quality validation,
    and both source batches complete successfully.
    """

    results = {
        "titles_batch_id": titles_batch_id,
        "credits_batch_id": credits_batch_id,
    }

    # Mark the overall pipeline and participating batches as active.
    mark_pipeline_running(
        PIPELINE_NAME
    )

    mark_batch_running(
        titles_batch_id
    )

    mark_batch_running(
        credits_batch_id
    )

    try:

        # Transform staged source data into the operational model.
        oltp_results = run_oltp_transformations(
            titles_batch_id=titles_batch_id,
            credits_batch_id=credits_batch_id,
        )

        results["oltp"] = oltp_results


        # Build the analytical warehouse from the updated OLTP state.
        olap_results = run_ola

### Verify Persisted Batch Lifecycle

Confirm that the titles and credits batches were marked as `SUCCESS` in `etl.batch_history` after the complete production pipeline finished.

This ensures that the database metadata agrees with the pipeline result returned by the coordinator.

In [21]:
# Inspect the persisted lifecycle state for both source batches.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:

        cursor.execute(
            """
            SELECT
                batch_id,
                file_name,
                status,
                processing_started_at,
                processing_completed_at,
                error_message
            FROM etl.batch_history
            WHERE batch_id IN (
                %(titles_batch_id)s,
                %(credits_batch_id)s
            )
            ORDER BY batch_id;
            """,
            {
                "titles_batch_id": titles_batch_id,
                "credits_batch_id": credits_batch_id,
            },
        )

        batch_lifecycle_results = cursor.fetchall()


# Display the persisted batch lifecycle metadata.
for row in batch_lifecycle_results:
    print(row)


# Confirm that both source batches completed successfully.
assert len(batch_lifecycle_results) == 2

assert all(
    row[2] == "SUCCESS"
    for row in batch_lifecycle_results
)

assert all(
    row[4] is not None
    for row in batch_lifecycle_results
)

assert all(
    row[5] is None
    for row in batch_lifecycle_results
)


print("PASS: batch lifecycle persisted successfully.")

(12, 'titles.csv', 'SUCCESS', datetime.datetime(2026, 8, 19, 9, 32, 53, 691171, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), datetime.datetime(2026, 8, 22, 0, 5, 49, 714854, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), None)
(13, 'credits.csv', 'SUCCESS', datetime.datetime(2026, 8, 19, 9, 37, 25, 321016, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), datetime.datetime(2026, 8, 22, 0, 5, 49, 738010, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), None)
PASS: batch lifecycle persisted successfully.


### Reload Pipeline Control Components

Reload the updated batch tracker, pipeline-control module, and production coordinator.

This ensures the notebook uses the latest lifecycle and checkpoint-management logic without restarting the entire kernel.

In [22]:
# Reload the updated production modules.
import importlib

import src.batch_tracker as batch_tracker
import src.pipeline_control as pipeline_control
import src.pipeline_runner as pipeline_runner


importlib.reload(
    batch_tracker
)

importlib.reload(
    pipeline_control
)

importlib.reload(
    pipeline_runner
)


# Rebind the updated coordinator.
run_transformation_pipeline = (
    pipeline_runner.run_transformation_pipeline
)

print("PASS: pipeline control components reloaded.")

PASS: pipeline control components reloaded.


### Load Pipeline Control Reader

Import the pipeline-control helper used to inspect the current pipeline checkpoint.

This allows the notebook to verify whether the successful production run correctly updated the persisted pipeline state.

In [23]:
# Import the pipeline-control reader used for checkpoint validation.
from src.pipeline_control import get_pipeline_control


print("PASS: pipeline control reader imported.")

PASS: pipeline control reader imported.


### Capture Pipeline Checkpoint Before Execution

Capture the current pipeline-control state before running the production pipeline again.

This provides a baseline that can be compared with the committed checkpoint after the successful execution.

In [24]:
# Read the current pipeline checkpoint before the next controlled run.
pipeline_control_before = get_pipeline_control(
    "netflix_incremental_pipeline"
)


print(
    "Pipeline control before:",
    pipeline_control_before,
)

assert pipeline_control_before is not None

print("PASS: baseline pipeline checkpoint captured.")

Pipeline control before: ('netflix_incremental_pipeline', datetime.datetime(2026, 8, 22, 0, 5, 49, 756849, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), datetime.datetime(2026, 8, 19, 9, 32, 53, 604618, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), 'titles.csv + credits.csv', 0, 'SUCCESS', datetime.datetime(2026, 8, 22, 0, 5, 49, 756849, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
PASS: baseline pipeline checkpoint captured.


### Execute Pipeline with Checkpoint Management

Run the complete production transformation pipeline again using the current source batches.

Because the transformations are idempotent, rerunning the same batches should preserve the data while allowing us to validate the pipeline-control lifecycle.

In [25]:
# Execute the complete controlled production pipeline.
pipeline_results = run_transformation_pipeline(
    titles_batch_id=titles_batch_id,
    credits_batch_id=credits_batch_id,
)


print(
    "Pipeline status:",
    pipeline_results["status"],
)

print(
    "Watermark:",
    pipeline_results["watermark"],
)

print(
    "Rows processed:",
    pipeline_results["rows_processed"],
)


assert pipeline_results["status"] == "SUCCESS"

print("PASS: controlled production pipeline completed.")

Pipeline status: SUCCESS
Watermark: 2026-08-19 09:32:53.604618+01:00
Rows processed: 0
PASS: controlled production pipeline completed.


### Verify Pipeline Checkpoint Advancement

Compare the pipeline-control state before and after the successful execution.

The pipeline must finish as `SUCCESS` and retain a valid successful-load timestamp and watermark. The checkpoint must only be committed after all transformation and quality stages have passed.

In [26]:
# Read the committed pipeline checkpoint after the successful run.
pipeline_control_after = get_pipeline_control(
    "netflix_incremental_pipeline"
)


print(
    "Pipeline control before:",
    pipeline_control_before,
)

print(
    "Pipeline control after:",
    pipeline_control_after,
)


# Confirm the pipeline-control record exists.
assert pipeline_control_after is not None

# Confirm the final pipeline state is successful.
assert pipeline_control_after[5] == "SUCCESS"

# Confirm that successful-load and watermark values exist.
assert pipeline_control_after[1] is not None
assert pipeline_control_after[2] is not None

# Confirm the control record was updated during this execution.
assert pipeline_control_after[6] >= pipeline_control_before[6]


print("PASS: pipeline checkpoint advanced successfully.")

Pipeline control before: ('netflix_incremental_pipeline', datetime.datetime(2026, 8, 22, 0, 5, 49, 756849, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), datetime.datetime(2026, 8, 19, 9, 32, 53, 604618, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), 'titles.csv + credits.csv', 0, 'SUCCESS', datetime.datetime(2026, 8, 22, 0, 5, 49, 756849, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
Pipeline control after: ('netflix_incremental_pipeline', datetime.datetime(2026, 8, 22, 0, 6, 12, 532163, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), datetime.datetime(2026, 8, 19, 9, 32, 53, 604618, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), 'titles.csv + credits.csv', 0, 'SUCCESS', datetime.datetime(2026, 8, 22, 0, 6, 12, 532163, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
PASS: pipeline checkpoint advanced successfully.


### Validate Failure Handling and Watermark Safety

Simulate a controlled pipeline failure and confirm that the previous successful checkpoint is preserved.

A failed run must mark the pipeline as `FAILED` without advancing `last_watermark`, `last_successful_load`, or the committed successful checkpoint.

In [27]:
# Import the production pipeline control reader.
from src.pipeline_control import get_pipeline_control


# Capture the last known successful checkpoint.
failure_test_checkpoint_before = get_pipeline_control(
    "netflix_incremental_pipeline"
)

print(
    "Checkpoint before failure test:",
    failure_test_checkpoint_before,
)

Checkpoint before failure test: ('netflix_incremental_pipeline', datetime.datetime(2026, 8, 22, 0, 6, 12, 532163, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), datetime.datetime(2026, 8, 19, 9, 32, 53, 604618, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), 'titles.csv + credits.csv', 0, 'SUCCESS', datetime.datetime(2026, 8, 22, 0, 6, 12, 532163, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))


### Simulate a Controlled Production Failure

Trigger the production coordinator with an invalid titles batch identifier.

This should cause the transformation pipeline to fail, update operational status to `FAILED`, and leave the last successful watermark unchanged.

In [28]:
# Use an invalid batch ID that does not exist in staging.
invalid_titles_batch_id = 999999999


try:

    run_transformation_pipeline(
        titles_batch_id=invalid_titles_batch_id,
        credits_batch_id=credits_batch_id,
    )

except Exception as error:

    print(
        "Expected pipeline failure:",
        type(error).__name__,
        "-",
        error,
    )


print(
    "PASS: controlled failure was triggered."
)

Expected pipeline failure: RuntimeError - Titles batch 999999999 was not found.
PASS: controlled failure was triggered.


### Verify Watermark Preservation After Failure

Inspect the pipeline-control record after the failed execution.

The pipeline status may change to `FAILED`, but the previously committed successful watermark and successful-load checkpoint must remain unchanged.

In [29]:
# Read pipeline control after the failure simulation.
failure_test_checkpoint_after = get_pipeline_control(
    "netflix_incremental_pipeline"
)


print(
    "Checkpoint before:",
    failure_test_checkpoint_before,
)

print(
    "Checkpoint after:",
    failure_test_checkpoint_after,
)


# Confirm the last successful checkpoint was preserved.
assert (
    failure_test_checkpoint_after[1]
    == failure_test_checkpoint_before[1]
)

assert (
    failure_test_checkpoint_after[2]
    == failure_test_checkpoint_before[2]
)

assert (
    failure_test_checkpoint_after[3]
    == failure_test_checkpoint_before[3]
)

assert (
    failure_test_checkpoint_after[4]
    == failure_test_checkpoint_before[4]
)


print(
    "PASS: failed run did not advance the successful checkpoint."
)

Checkpoint before: ('netflix_incremental_pipeline', datetime.datetime(2026, 8, 22, 0, 6, 12, 532163, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), datetime.datetime(2026, 8, 19, 9, 32, 53, 604618, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), 'titles.csv + credits.csv', 0, 'SUCCESS', datetime.datetime(2026, 8, 22, 0, 6, 12, 532163, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
Checkpoint after: ('netflix_incremental_pipeline', datetime.datetime(2026, 8, 22, 0, 6, 12, 532163, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), datetime.datetime(2026, 8, 19, 9, 32, 53, 604618, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')), 'titles.csv + credits.csv', 0, 'FAILED', datetime.datetime(2026, 8, 22, 0, 8, 26, 407127, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
PASS: failed run did not advance the successful checkpoint.


### Reload Batch Validation Components

Reload the updated batch tracker and production pipeline coordinator.

This ensures the notebook uses the new precondition logic that rejects invalid batches before any transformation begins.

In [31]:
# Reload the modules changed during batch-precondition development.
import importlib

import src.batch_tracker as batch_tracker
import src.pipeline_runner as pipeline_runner


importlib.reload(
    batch_tracker
)

importlib.reload(
    pipeline_runner
)


# Rebind the updated production coordinator.
run_transformation_pipeline = (
    pipeline_runner.run_transformation_pipeline
)

print("PASS: batch validation components reloaded.")

PASS: batch validation components reloaded.


### Validate Batch Preconditions

Confirm that the production pipeline rejects an invalid batch before any OLTP or OLAP transformation begins.

This provides a clear failure boundary for missing source metadata.

In [32]:
# Use a batch identifier that does not exist.
invalid_batch_id = 999999999


try:

    run_transformation_pipeline(
        titles_batch_id=invalid_batch_id,
        credits_batch_id=credits_batch_id,
    )

except ValueError as error:

    print(
        "Expected validation failure:",
        error,
    )

else:

    raise AssertionError(
        "Invalid batch was not rejected."
    )


print("PASS: invalid batch rejected before transformation.")

Expected validation failure: Batch 999999999 does not exist.
PASS: invalid batch rejected before transformation.
